# Dimensionality Reduction of scRNA-seq Data with scVI

CSUF EAGER — single-cell RNA-seq module

A single-cell experiment measures thousands of genes across thousands of cells,
so every cell starts as a point in a ~20,000-dimensional space. Most of that is
sparsity and technical noise, not biology. We'll use **scVI** to learn a compact
latent representation that keeps the biological structure, then use it to tell
apart **cell types** (what a cell *is*, e.g. CD4 vs CD8 T cell) from **cell
states** (what a cell is *doing*, e.g. resting, proliferating, exhausted).

**The dataset** (CAR-T fitness, experiment 17) is mouse plate-based scRNA-seq
of Her2-targeting CAR-T cells, either untreated (`noTx`) or treated with an
IDH2 inhibitor (`IDH2i`). Cells were sorted from:

- the **infusion product** (CAR-T cells before injection)
- **co-culture** with Her2+ tumor cells in a dish
- tumors and spleens of treated mice: the injected **donor** CAR-T cells, plus
  the host's own T cells (`host_T`) and immune cells (`host_CD45`)

## Install

Run once. Restart the kernel afterwards, then skip this cell for the rest of the
session.

In [ ]:
# pynndescent 0.7 breaks scanpy 1.12 neighbors ("'csr_array' object has no attribute 'getnnz'")
%pip install -q scvi-tools scanpy leidenalg igraph "pynndescent<0.7"

In [ ]:
import warnings
from importlib.metadata import version

# Known-harmless warnings: older optional pandas helpers, and a GPU too old for this PyTorch build
for msg in [".*numexpr", ".*bottleneck", ".*does not include kernels", ".*CUDA capability"]:
    warnings.filterwarnings("ignore", message=msg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import torch

for pkg in ["scanpy", "scvi-tools", "torch"]:
    print(f"{pkg:11}", version(pkg))

# A GPU is only usable if this PyTorch build was compiled for its architecture
USE_GPU = False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    USE_GPU = f"sm_{major}{minor}" in torch.cuda.get_arch_list()
    note = "" if USE_GPU else " — too old for this PyTorch build, using CPU"
    print("GPU        ", torch.cuda.get_device_name(0) + note)
else:
    print("GPU         none (CPU)")
ACCELERATOR = "gpu" if USE_GPU else "cpu"

scvi.settings.seed = 0
sc.set_figure_params(dpi=80, facecolor="white")

## Paths and settings

The trained model weights ship with the repo in `data/scvi_model/` (and are
downloaded automatically if missing), so in class we **load** the model instead
of training it. Set `RETRAIN = True` to train from scratch (a few minutes on a
GPU, longer on CPU).

In [ ]:
import urllib.request
from pathlib import Path

H5AD      = "CART_fitness_exp17_combined_qc_scvi_flow_ct2.h5ad"
MODEL_URL = "https://raw.githubusercontent.com/CSUF-EAGER/EAGER_scRNA-seq/main/data/scvi_model/model.pt"

# data/ is next to the notebook (README download) or one level up (git clone, notebook in scripts/)
DATA_DIR  = next((d for d in [Path("data"), Path("../data")] if (d / H5AD).exists()), Path("data"))
DATA      = DATA_DIR / H5AD
MODEL_DIR = DATA_DIR / "scvi_model"

RETRAIN = False

if not RETRAIN and not (MODEL_DIR / "model.pt").exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(MODEL_URL, MODEL_DIR / "model.pt")

print(DATA, "found" if DATA.exists() else "MISSING — see README")
print(MODEL_DIR / "model.pt", "found" if (MODEL_DIR / "model.pt").exists() else "missing — will train")

## 1. Load the data into an `AnnData` object

`AnnData` is the standard container for single-cell data:

| slot | contents |
|---|---|
| `adata.X` | the cells × genes matrix |
| `adata.obs` | one row per **cell** (sample, annotations, QC metrics) |
| `adata.var` | one row per **gene** |
| `adata.layers` | alternative versions of `X` (e.g. raw counts) |
| `adata.obsm` | per-cell embeddings (PCA, scVI latent, UMAP) |

In [ ]:
adata = sc.read_h5ad(DATA)
adata

In [ ]:
adata.obs.head()

This file already contains results from a previous analysis (an `X_scVI`
embedding, Leiden clusters, and cell annotations). We'll recompute the
embedding ourselves and use the annotations as the "answer key".

`cell_type_v2` is a fine-grained label that mixes two kinds of information:

- **cell type**, i.e. lineage: is it a T cell, NK cell, macrophage, dendritic cell or B cell?
- **cell state**, i.e. which program a T cell is running: naive (`Tnaive`), stem-like
  memory (`Tscm`, `Tcm`), effector (`Teff`), progenitor-exhausted (`Tpex`),
  terminally exhausted (`Tterm`), cycling, and so on.

We split it into two columns: a broad `cell_lineage` (type) and the fine label
(state).

In [ ]:
GROUP_KEY      = "gate"          # where the cells were sorted from
CONDITION_KEY  = "treatment"     # IDH2i vs noTx
CELL_STATE_KEY = "cell_type_v2"  # fine-grained state
CELL_TYPE_KEY  = "cell_lineage"  # broad type, built below

lineage = {
    "T cell":    ["Tnaive", "Tscm", "Tcm", "Teff_early", "Teff", "Tpex", "Tpex_int",
                  "Tpex_cycling", "Tex_cycling", "Tterm", "CD4_Teff", "Treg"],
    "NK cell":   ["NK", "NK_act"],
    "Myeloid":   ["TAM_homeo", "TAM_inflam"],
    "Dendritic": ["cDC1", "cDC2", "mDC_Ccr7", "pDC"],
    "B cell":    ["Bcell"],
}
state_to_lineage = {state: lin for lin, states in lineage.items() for state in states}
adata.obs[CELL_TYPE_KEY] = pd.Categorical(adata.obs[CELL_STATE_KEY].map(state_to_lineage),
                                          categories=list(lineage))

# Order the T-cell states roughly from naive to exhausted, so plots read left to right
state_order = [s for states in lineage.values() for s in states]
adata.obs[CELL_STATE_KEY] = adata.obs[CELL_STATE_KEY].cat.reorder_categories(state_order)

for key in [GROUP_KEY, CONDITION_KEY, CELL_TYPE_KEY, CELL_STATE_KEY]:
    print(f"--- {key}")
    print(adata.obs[key].value_counts().to_string(), "\n")

## 2. Quality control

This file has already been filtered (`qc` in the filename), and the QC metrics
are stored in `adata.obs`. Here we just look at them to see what "good" cells
look like:

- `n_genes_by_counts`: genes detected per cell (too low means an empty well; too high, a doublet)
- `total_counts`: reads (UMIs) per cell
- `pct_counts_mt`: percentage of reads from mitochondrial genes (high means a dying cell)

scVI models **raw counts**, so we also check that the `counts` layer holds
integers. (`adata.X` here is already normalized and log-transformed.)

In [ ]:
COUNTS_LAYER = "counts"
counts = adata.layers[COUNTS_LAYER]
print("integer counts:", np.allclose(counts.data, np.round(counts.data)), "| max:", counts.max())
print("X is log-normalized, max:", round(float(adata.X.max()), 2))

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             groupby=GROUP_KEY, rotation=45, multi_panel=True)

One more filter: 140 cells were flagged as contaminants from the wrong sort
gate (`excluded_gate_contam`) and never got an annotation. We drop them.

In [ ]:
print(adata.obs["excluded_gate_contam"].value_counts().to_string())
adata = adata[~adata.obs["excluded_gate_contam"]].copy()
adata

## 3. Highly variable gene selection

Most genes barely change between cells and only add noise. This file has
already been trimmed to 5,000 genes; we keep the 2,000 whose expression varies
most from cell to cell.

This step is deterministic, so it picks the same genes on every machine. That is
what lets us load the pre-trained model later.

In [ ]:
N_HVG = 2000

sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=N_HVG,
                            layer=COUNTS_LAYER, subset=False)
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg

## 4. Train scVI

scVI is a **variational autoencoder**:

- the **encoder** squeezes each cell's ~2,000 gene counts down to 10 numbers (the latent space)
- the **decoder** tries to rebuild the counts from those 10 numbers, using a
  negative-binomial model that fits the noise in count data

scVI can also correct for **batch effects** if you give it a `batch_key`, for
example the sorting session. But a batch key only helps when every batch
contains the same kinds of cells. Check this:

In [ ]:
pd.crosstab(adata.obs["sorting_session"], adata.obs["experiment_type"])

Every in-vivo cell came from one session, and the other two sessions hold only
in-vitro cells. Session and biology are **confounded**: "correcting" for session
would also erase the real differences between in-vivo and in-vitro cells. So we
train without a batch key.

The weights in `data/scvi_model/` were trained with this exact cell. If they
exist and `RETRAIN` is `False`, we just load them.

In [ ]:
scvi.model.SCVI.setup_anndata(adata_hvg, layer=COUNTS_LAYER)

if RETRAIN or not (MODEL_DIR / "model.pt").exists():
    model = scvi.model.SCVI(adata_hvg, n_latent=10, n_layers=2, gene_likelihood="nb")
    model.train(max_epochs=400, early_stopping=True, accelerator=ACCELERATOR)
    model.save(MODEL_DIR, overwrite=True, save_anndata=False)
    print("trained and saved to", MODEL_DIR)
else:
    model = scvi.model.SCVI.load(MODEL_DIR, adata=adata_hvg, accelerator=ACCELERATOR)
    print("loaded pre-trained model from", MODEL_DIR)

model

The training curve: the loss (ELBO) should drop quickly and then flatten out. If
the validation curve climbs back up while training keeps falling, the model is
overfitting.

In [ ]:
hist = model.history
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(hist["elbo_train"], label="train")
if "elbo_validation" in hist:
    ax.plot(hist["elbo_validation"], label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("ELBO loss"); ax.legend()
plt.show()

## 5. Extract the latent space

Each cell is now 10 numbers instead of ~2,000.

In [ ]:
adata.obsm["X_scVI"] = model.get_latent_representation(adata_hvg)
adata.obsm["X_scVI"].shape

## 6. UMAP: scVI latent space vs. plain PCA

UMAP squashes a high-dimensional space down to 2-D for plotting while keeping
neighboring cells together. It is only as good as the space it's given, so we
build one UMAP from scVI's latent space and one from the standard
normalize → log → PCA pipeline, and compare them.

In [ ]:
# Baseline: library-size normalize, log-transform, PCA on the same HVGs
pca = adata_hvg.copy()
pca.X = pca.layers[COUNTS_LAYER].copy()
pca.uns.pop("log1p", None)  # X is raw counts again; drop the "already logged" flag
sc.pp.normalize_total(pca, target_sum=1e4)
sc.pp.log1p(pca)
sc.pp.pca(pca, n_comps=30)
sc.pp.neighbors(pca, use_rep="X_pca")
sc.tl.umap(pca)
adata.obsm["X_umap_pca"] = pca.obsm["X_umap"]

# scVI
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"]

In [ ]:
for basis, title in [("umap_pca", "PCA"), ("umap_scvi", "scVI")]:
    sc.pl.embedding(adata, basis=basis, color=[GROUP_KEY, CELL_TYPE_KEY, CELL_STATE_KEY],
                    title=[f"{title}: sort gate", f"{title}: cell type", f"{title}: cell state"],
                    wspace=0.5, frameon=False)

**What to look for:**

- **Cell type:** the lineages (T, NK, myeloid, DC, B) should form clearly
  separate islands. Different lineages express very different genes, so this
  is the easy part.
- **Cell state:** T-cell states sit *inside* the big T-cell region, often as a
  continuum (naive → effector → exhausted) rather than separate islands. Is
  that continuum cleaner in scVI or in PCA?
- **Sort gate:** infusion-product and co-culture cells sit apart from the
  in-vivo cells, partly from real biology (a dish vs a tumor) and partly from
  technical differences between sorting sessions, which we couldn't correct
  for.

## 7. Clustering

Leiden clustering groups cells that are close together in the scVI neighbor
graph. The clusters come from the data alone, with no labels involved; next
we'll check how well they match the annotations.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, key_added="leiden", flavor="igraph", n_iterations=2)
sc.pl.embedding(adata, basis="umap_scvi", color="leiden", legend_loc="on data", frameon=False)

## 8. Bar plots: composition of types and states

UMAPs are useful for eyeballing, but the proportions are easier to read off bar
plots.

In [ ]:
def stacked_bar(df, index, columns, title, ax):
    """Stacked bar of the fraction of each `columns` category within each `index` group."""
    frac = pd.crosstab(df[index], df[columns], normalize="index")
    frac.plot.bar(stacked=True, ax=ax, width=0.85, legend=False)
    ax.set_ylabel("fraction of cells"); ax.set_xlabel(""); ax.set_title(title)
    ax.legend(title=columns, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

fig, ax = plt.subplots(figsize=(7, 4.5))
stacked_bar(adata.obs, GROUP_KEY, CELL_TYPE_KEY, "Cell types per sort gate", ax)
plt.tight_layout(); plt.show()

The `host_CD45` gate catches every host immune cell, so it's a mix of lineages;
the CAR-T gates are almost entirely T cells. Now zoom in on T cells only and
look at their **states**:

In [ ]:
tcells = adata.obs[adata.obs[CELL_TYPE_KEY] == "T cell"].copy()
tcells[CELL_STATE_KEY] = tcells[CELL_STATE_KEY].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(7, 4.5))
stacked_bar(tcells, GROUP_KEY, CELL_STATE_KEY, "T-cell states per sort gate", ax)
plt.tight_layout(); plt.show()

Same T cells, the same cell *type*, but very different *states* depending on
where they came from. The infusion product is mostly stem-like (`Tscm`); after
time in a tumor, the donor CAR-T cells are dominated by exhausted states
(`Tterm`, `Tpex`, `Tex_cycling`).

The research question is whether the **IDH2 inhibitor** keeps CAR-T cells
fitter. Split each gate by treatment:

In [ ]:
tcells["gate_treatment"] = tcells[GROUP_KEY].astype(str) + " | " + tcells[CONDITION_KEY].astype(str)

fig, ax = plt.subplots(figsize=(9, 4.5))
stacked_bar(tcells, "gate_treatment", CELL_STATE_KEY, "T-cell states by gate and treatment", ax)
plt.setp(ax.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")
plt.tight_layout(); plt.show()

And what each Leiden cluster is made of. A cluster made of one cell type but
mixed states means scVI separated identity first; a cluster made of one state
across several types means the state signal is the stronger one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
stacked_bar(adata.obs, "leiden", CELL_TYPE_KEY, "Cell types per cluster", axes[0])
stacked_bar(adata.obs, "leiden", CELL_STATE_KEY, "Cell states per cluster", axes[1])
plt.tight_layout(); plt.show()

## 9. Marker genes

Finally, the genes behind the separation. This is mouse data, so gene names are
written `Cd4`, not `CD4`. (A few classic markers, e.g. `Cd8a` and `Tcf7`, were
dropped when the file was trimmed to 5,000 genes.)

**Type markers** distinguish lineages:

In [ ]:
type_markers = {
    "T":       ["Cd4", "Foxp3", "Il7r"],
    "NK":      ["Klrb1c", "Ncr1"],
    "Myeloid": ["C1qa", "Adgre1", "Lyz2", "S100a8"],
    "DC":      ["Xcr1", "Clec9a", "Fscn1", "Siglech"],
    "B":       ["Cd19", "Ms4a1"],
}

# adata.X is already normalized + log-transformed, which is what we want for plotting
sc.pl.dotplot(adata, type_markers, groupby=CELL_TYPE_KEY, standard_scale="var")

**State markers** distinguish what T cells are doing:

| state | genes | meaning |
|---|---|---|
| naive / memory | `Ccr7`, `Sell`, `Il7r`, `Id3`, `Slamf6` | resting, long-lived, "fit" |
| activated | `Cd69`, `Nr4a1`, `Tnfrsf9` | just stimulated |
| interferon | `Ifit1`, `Ifit3`, `Isg15` | responding to interferon |
| effector | `Gzma`, `Gzmb`, `Prf1`, `Klrg1`, `Cx3cr1` | actively killing |
| exhausted | `Pdcd1`, `Lag3`, `Tox`, `Tigit`, `Cd160` | chronically stimulated, worn out |
| cycling | `Mki67`, `Top2a`, `Stmn1` | dividing |

In [ ]:
state_markers = {
    "naive/mem": ["Ccr7", "Sell", "Il7r", "Id3", "Slamf6"],
    "activated": ["Cd69", "Nr4a1", "Tnfrsf9"],
    "IFN":       ["Ifit1", "Ifit3", "Isg15"],
    "effector":  ["Gzma", "Gzmb", "Prf1", "Klrg1", "Cx3cr1"],
    "exhausted": ["Pdcd1", "Lag3", "Tox", "Tigit", "Cd160"],
    "cycling":   ["Mki67", "Top2a", "Stmn1"],
}

t_adata = adata[adata.obs[CELL_TYPE_KEY] == "T cell"]
sc.pl.dotplot(t_adata, state_markers, groupby=CELL_STATE_KEY, standard_scale="var")

In [ ]:
sc.pl.embedding(adata, basis="umap_scvi", color=["Cd4", "C1qa", "Sell", "Gzmb", "Pdcd1", "Mki67"],
                ncols=3, frameon=False, cmap="viridis")

## Takeaways

- scVI compresses 2,000 genes per cell into 10 latent dimensions by modeling the
  raw counts directly.
- Batch correction is only safe when batches aren't confounded with biology.
- Cell **type** shows up as separate islands in the UMAP; cell **state** shows
  up as gradients and sub-regions *within* a type. The bar plots turn that into
  numbers you can compare across conditions.

**Try it:** change `n_latent`, `N_HVG`, or the Leiden `resolution`, set
`RETRAIN = True`, and see how the UMAP and bar plots change.